# 第一阶段：Olist 订单数据读取与清洗

本阶段的清洗逻辑已抽离到 `scripts/data_cleaning.py`，Notebook 只保留**调用**与**结果查看**，
不再出现大段未封装的清洗代码。

`process_orders()` 函数内部完成：

1. 读取原始文件 `data/raw/olist_orders_dataset.csv`；
2. 将 5 个时间字段统一转换为 `datetime`；
3. 检查并处理缺失值（业务上合理的空值保留为 `NaT`）；
4. 输出清洗结果到 `data/processed/orders_cleaned.csv`，并写入 SQLite 数据库 `data/olist.db` 的 `orders_cleaned` 表。


In [1]:
# 读取、清洗（时间格式转换 / 缺失值处理）、存 CSV 与入库，全部封装在脚本里
# 脚本内部按项目根目录解析路径，因此在 Notebook 里调用也能正确定位 data/
from scripts.data_cleaning import process_orders

orders = process_orders()


原始数据规模：99441 行 × 8 列
order_id 是否存在重复：False

处理前缺失值统计：
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64
各时间字段缺失率（%，保留空值）：
order_purchase_timestamp         0.00
order_approved_at                0.16
order_delivered_carrier_date     1.79
order_delivered_customer_date    2.98
order_estimated_delivery_date    0.00
dtype: float64



已保存清洗后 CSV：data/processed/orders_cleaned.csv


已写入 SQLite：data/olist.db -> orders_cleaned（99,441 行，与 DataFrame 一致：True）


In [2]:
# 查看前 5 行
orders.head()


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [3]:
# 查看数据结构与字段类型
orders.info()


<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


# 第二阶段：其余 5 个核心表清洗入库

本阶段用**配置驱动 + 统一函数**的方式批量处理支付、订单明细、评论、商品、品类翻译 5 张表，
避免为每张表重复写 5 遍代码。清洗逻辑已抽离到 `scripts/data_cleaning.py`，
Notebook 只保留调用与入库校验。

## 处理流程（`clean_and_load` 函数，每张表执行一次）

1. 按 `data/raw/<原始文件名>` 读取 CSV；
2. 对配置中声明的时间字段执行 `pd.to_datetime(..., errors="coerce")`，无法解析的值转为 `NaT`；
3. 用 `drop_duplicates()` 剔除**完全重复**的行（所有字段取值都相同才算重复，
   不会误删同一订单的多条支付记录或明细行）；
4. 以 `utf-8-sig` 编码输出到 `data/processed/<表名>_cleaned.csv`；
5. 通过 SQLAlchemy 引擎写入 `data/olist.db`，`if_exists="replace"` 保证重复运行安全（整表覆盖）；
6. 打印入库结果，包含行数、列数、去重行数与 CSV 路径。

表名与时间字段在脚本的 `TABLES` 配置中集中声明，新增表只需追加一条配置。

## 表清单与处理结果

| 原始文件 | 数据库表名 | 行数 | 列数 | 去重行数 | 转换为 datetime 的字段 |
| --- | --- | ---: | ---: | ---: | --- |
| `olist_order_payments_dataset.csv` | `payments` | 103,886 | 5 | 0 | 无（无时间字段） |
| `olist_order_items_dataset.csv` | `order_items` | 112,650 | 7 | 0 | `shipping_limit_date` |
| `olist_order_reviews_dataset.csv` | `reviews` | 99,224 | 7 | 0 | `review_creation_date`、`review_answer_timestamp` |
| `olist_products_dataset.csv` | `products` | 32,951 | 9 | 0 | 无（均为数值/文本字段） |
| `product_category_name_translation.csv` | `category_translation` | 71 | 2 | 0 | 无（品类名称映射表） |

> 原始数据本身不含完全重复行，本阶段去重命中 0 行；保留 `drop_duplicates()` 是为了在数据源更新或
> 多源合并时自动兜底。

## 产出物

- `data/processed/` 下新增 5 个清洗结果文件：
  `payments_cleaned.csv`、`order_items_cleaned.csv`、`reviews_cleaned.csv`、
  `products_cleaned.csv`、`category_translation_cleaned.csv`。
- `data/olist.db` 现有 6 张表：`orders_cleaned`（第一阶段）、`payments`、`order_items`、
  `reviews`、`products`、`category_translation`。
- 入库后由校验单元格重新读取数据库，核对行数与字段，确认与 DataFrame 一致；
  时间字段在 SQLite 中以 `YYYY-MM-DD HH:MM:SS.ffffff` 文本存储，可直接用字符串比较或 `strftime()` 处理。

## 注意事项

1. `reviews` 的评论标题与评论内容存在大量空值，属于用户未填写，保持 `NaN` 不做填充。
2. `products` 原始列名存在拼写错误（`product_name_lenght`、`product_description_lenght`），
   清洗时**保持原样**以免破坏下游脚本，如需修正建议在分析阶段显式 `rename`。
3. `products` 的品类字段有空值，后续与 `category_translation` 关联时需注意空品类无法翻译。

## 一次性重跑 6 张表

如果需要把第一、第二阶段一起跑完（例如换了新数据源），直接调用主函数即可：

```python
from scripts.data_cleaning import process_all_data

cleaned = process_all_data()   # 返回 {"orders_cleaned": df, "payments": df, ...}
```

也可以在命令行独立执行 `python scripts/data_cleaning.py`，会连带打印入库校验结果。


In [4]:
# 阶段二：配置驱动批量清洗其余 5 张表（时间字段转换 / 去重 / 存 CSV / 入库）
from scripts.data_cleaning import process_secondary_tables

cleaned_tables = process_secondary_tables()


表 payments 已成功入库：103,886 行 × 5 列（原始 103,886 行，去重 0 行），CSV -> data/processed/payments_cleaned.csv


表 order_items 已成功入库：112,650 行 × 7 列（原始 112,650 行，去重 0 行），CSV -> data/processed/order_items_cleaned.csv


表 reviews 已成功入库：99,224 行 × 7 列（原始 99,224 行，去重 0 行），CSV -> data/processed/reviews_cleaned.csv


表 products 已成功入库：32,951 行 × 9 列（原始 32,951 行，去重 0 行），CSV -> data/processed/products_cleaned.csv
表 category_translation 已成功入库：71 行 × 2 列（原始 71 行，去重 0 行），CSV -> data/processed/category_translation_cleaned.csv

全部 5 张表处理完成。


In [5]:
# 入库校验：重新从数据库读取，核对每张表的行数、字段与时间字段样例
from scripts.data_cleaning import verify_database

verify_database()


入库校验：data/olist.db
orders_cleaned       行数= 99,441  字段=['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
  order_purchase_timestamp          order_approved_at order_delivered_carrier_date order_delivered_customer_date order_estimated_delivery_date
2017-10-02 10:56:33.000000 2017-10-02 11:07:15.000000   2017-10-04 19:55:00.000000    2017-10-10 21:25:13.000000    2017-10-18 00:00:00.000000
2018-07-24 20:41:37.000000 2018-07-26 03:24:27.000000   2018-07-26 14:31:00.000000    2018-08-07 15:27:45.000000    2018-08-13 00:00:00.000000

payments             行数=103,886  字段=['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

order_items          行数=112,650  字段=['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
       shipping_limit_date
2017-09-19 09:45:35.000000

## 第三阶段：配置 JupySQL —— 在 Notebook 中直接写 SQL

前面两个阶段已经把 6 张核心表清洗进 `data/olist.db`。从这一阶段开始，为了做复杂的业务分析（销售额、复购、差评归因等），
我们不再用 pandas 拼 DataFrame，而是**直接写 SQL**。

**做法**：用 `jupysql` 扩展（`%load_ext sql`）把 SQLite 数据库注册成 JupySQL 的连接，
之后每个单元格都可以用魔术命令执行 SQL：

| 写法 | 作用 | 能否给 Python 变量赋值 |
| --- | --- | --- |
| `%sql SELECT ...`（行魔术，单元格内只是一行 SQL） | 执行单条 SQL | ✅ `result = %sql SELECT ...` |
| `%%sql`（单元格魔术，放在单元格第一行） | 整个单元格都当 SQL 写，适合多行长查询 | ❌ 结果只打印 |

**本阶段产出**：可用 SQL 查询的 6 张表 + 查询结果转 DataFrame 的固定套路。


In [6]:
# 加载 JupySQL 扩展：之后本 Notebook 就可以使用 %sql / %%sql 魔术命令
# 只需在会话开始时执行一次；重启内核后需要重新执行
%load_ext sql

print("JupySQL 扩展加载完成")


JupySQL 扩展加载完成


In [7]:
# 连接本地 SQLite 数据库（路径相对于 Notebook 所在目录）
# 连接是"会话级"的：执行成功后，后续单元格直接用 %sql 即可，不用重复连接
%sql sqlite:///data/olist.db


Connecting to 'sqlite:///data/olist.db'

In [8]:
# 验证连接是否成功：查询 SQLite 的系统表，看 6 张核心表是否都在
%sql SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name;


Running query in 'sqlite:///data/olist.db'

name
category_translation
order_items
orders_cleaned
payments
products
reviews


In [9]:
# 用行魔术命令执行 SQL，并把结果赋值给 Python 变量
# 注意：%sql 那一行不是合法的 Python 语法，但 JupySQL 会在执行前把它替换掉
result = %sql SELECT order_status, COUNT(*) AS order_count FROM orders_cleaned GROUP BY order_status ORDER BY order_count DESC;

# 查询返回的是一个 ResultSet 对象，不是 DataFrame
print("查询返回类型：", type(result))

# 转成 pandas DataFrame，之后就能继续用 pandas 画图 / 计算
status_df = result.DataFrame()
print("转换后类型：", type(status_df))
status_df


Running query in 'sqlite:///data/olist.db'

查询返回类型： <class 'sql.run.resultset.ResultSet'>
转换后类型： <class 'pandas.DataFrame'>


,order_status,order_count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


### 补充说明：VS Code 里的操作方式

**1. 把单元格语言从 Python 切换到纯 SQL（VS Code）**

- 没有默认快捷键。点击单元格**右下角的语言标识**（显示为 `Python`）→ 在弹出列表里选 `SQL`；
  或者按 `Ctrl+Shift+P` 打开命令面板 → 输入 `Change Cell Language` → 选择 `SQL`。
- ⚠️ 但语言选择器只列出**当前内核支持的语言**。我们用 Python 内核（ipykernel），
  可选项只有 `Python` / `Markdown` / `Raw`，**没有 SQL**。也就是说"纯 SQL 单元格"在 Python 内核下跑不了。
- 所以实际可用的两种姿势：
  - 想要语法高亮 + 能赋值给变量 → Python 单元格里写一行 `%sql SELECT ...`；
  - 想要整段 SQL 长查询不转义 → Python 单元格第一行写 `%%sql`，下面随便写多少行 SQL。
- 如果确实想要"纯 SQL 单元格"，需要装一个 SQL 内核（如 `pip install jupysql-plugin` 配套的
  JupySQL VS Code 扩展）并把它选为 Notebook 内核，这属于另一套配置，本项目不依赖它。

**2. 常用 Notebook 快捷键（命令模式，先按 `Esc`）**

| 快捷键 | 作用 |
| --- | --- |
| `Esc` / `Enter` | 进入命令模式 / 编辑模式 |
| `M` / `Y` | 切换为 Markdown / 切换为代码 |
| `Ctrl+Enter` | 运行当前单元格 |
| `Shift+Enter` | 运行并选中下一个单元格 |
| `Alt+Enter` | 运行并在下方插入新单元格 |
| `A` / `B` | 在上方 / 下方插入单元格 |
| `dd` | 删除当前单元格 |

**3. `ResultSet` 常用方法**

```python
result = %sql SELECT * FROM orders_cleaned LIMIT 5
result.DataFrame()   # → pandas.DataFrame
result.dict()        # → 字典列表，便于 json 序列化
result.keys()        # → 列名
result.scalar()      # → 单个标量值（如 SELECT COUNT(*)）
```


## 第四阶段：业务指标 SQL 视图（月度 GMV / 有效订单数）

指标口径与踩过的坑都沉淀在 `sql/01_metrics_views.sql` 里，由 Notebook 调用执行，双向可复现。

视图 `v_monthly_gmv` 的三列：

| 列名 | 含义 |
| --- | --- |
| `order_month` | 下单月份，格式 `YYYY-MM`（取自 `order_purchase_timestamp`） |
| `valid_orders` | 当月有效订单数（`order_id` 去重） |
| `gmv` | 当月支付总额（`payments.payment_value` 按订单聚合后求和） |

**口径说明**

- **有效订单**：剔除 `canceled` / `unavailable`。这两类共 1,234 笔订单仍有支付流水，但不产生真实成交，不计入 GMV。
- **必须先聚合再 JOIN**：一个订单的 `payments` 可能有 1~29 行（信用卡 + 代金券 + 分期），
  不预聚合就直接 JOIN 会把订单行放大，订单数与金额一起失真。
- **支付口径**：`payment_value` 含运费（实测 99.42% 的订单满足 `payment_value = 商品价 + 运费`），
  属于"实收 GMV"，与只算商品价的"商品 GMV"不同。
- **数据完整性**：全期 24 个月。2016-09 / 2016-10 / 2016-12 是平台测试期尾单，
  2018-09 之后数据截断（2018-10 的 4 单全部为 `canceled`），做同比/环比时应先过滤 `BETWEEN '2017-01' AND '2018-08'`。

**如何在 Notebook 中执行这个 SQL 文件**

- 官方写法 `%sql --file ./sql/01_metrics_views.sql`：JupySQL 内部用 `Path(...).read_text()`（不指定编码），
  Windows 默认 GBK 读 UTF-8 中文注释会直接抛 `UnicodeDecodeError`。
- 下面的单元格改用「显式 UTF-8 读取 + `%sql {{ ddl }}` 模板变量」，任何 locale 下都能跑，推荐这种。
- `%run ./sql/01_metrics_views.sql` **不可用**：`%run` 是 IPython 的 Python 脚本执行器，
  会按 Python 语法解析 `.sql` 文件，实测报 `SyntaxError`。


In [10]:
from pathlib import Path

# 显式指定 UTF-8：Windows 默认 GBK，直接 %sql --file 会因中文注释报 UnicodeDecodeError
ddl = Path("sql/01_metrics_views.sql").read_text(encoding="utf-8")

# 交给 JupySQL 执行；文件里是 DROP VIEW + CREATE VIEW 两条语句，JupySQL 会按分号自动拆分
%sql {{ ddl }}

print("视图 v_monthly_gmv 已创建（DROP IF EXISTS + CREATE，可重复执行）")


Running query in 'sqlite:///data/olist.db'

视图 v_monthly_gmv 已创建（DROP IF EXISTS + CREATE，可重复执行）


In [11]:
# JupySQL 默认只显示 10 行（displaylimit），放宽后一次看完 24 个月
%config SqlMagic.displaylimit = 100

# 查询视图：24 个月的有效订单数与 GMV
%sql SELECT order_month, valid_orders, gmv FROM v_monthly_gmv ORDER BY order_month;


Running query in 'sqlite:///data/olist.db'

order_month,valid_orders,gmv
2016-09,2,136.23
2016-10,293,51657.53
2016-12,1,19.62
2017-01,787,137006.76
2017-02,1718,283621.94
2017-03,2617,425656.4
2017-04,2377,405988.38
2017-05,3640,582926.16
2017-06,3205,499827.47
2017-07,3946,578858.58


In [12]:
# 赋值给 Python 变量 → 转 DataFrame，后续可直接用 pandas 画趋势图
gmv_df = %sql SELECT order_month, valid_orders, gmv FROM v_monthly_gmv ORDER BY order_month
gmv_df = gmv_df.DataFrame()

print(f"月份数：{len(gmv_df)}")
print(f"全期 GMV：{gmv_df['gmv'].sum():,.2f} ｜ 全期有效订单：{gmv_df['valid_orders'].sum():,}")
gmv_df.tail()


Running query in 'sqlite:///data/olist.db'

月份数：24
全期 GMV：15,739,137.01 ｜ 全期有效订单：98,207


,order_month,valid_orders,gmv
19,2018-05,6833,1145748.63
20,2018-06,6145,1020494.29
21,2018-07,6233,1039880.16
22,2018-08,6421,996896.15
23,2018-09,1,166.46


## 第五阶段：专题分析 —— 卖家履约质量与差评率的关联

对应目标岗位（营销数据分析）的核心诉求：**把"履约时效"这个运营动作，量化成"差评率"这个业务结果**，
从而给出可执行的改进建议，而不是只罗列指标。

分析链路与口径（完整 SQL 见 `sql/02_seller_analysis.sql`）：

| 环节 | 表 | 关键处理 |
|---|---|---|
| 商品明细 | `order_items` | 先聚合到「订单 × 卖家」粒度（1,278 个多卖家订单，占 1.3%） |
| 履约时效 | `orders_cleaned` | 发货时长 = `order_delivered_carrier_date` − `order_purchase_timestamp`，单位小时 |
| 客户评价 | `reviews` | 547 个订单有多条评论（202 个分数互相矛盾），取最新回复的那条 |

三个必须交代的数据质量决策：

1. **评论去重**：同一订单取 `review_answer_timestamp` 最新的一条，否则重复评论会让差评率被重复计数。
2. **异常时长剔除**：166 单"揽收时间早于下单时间"（负值，系统录入错误）、6 单超过 60 天，均不参与均值；
   有效区间取 0 ~ 1440 小时。
3. **卖家样本门槛**：571 个卖家只有 1 单、872 个只有 2~5 单，这类卖家差评率非 0 即 1，
   会把四分位边界拉向极端值；只保留 **≥ 10 单**的卖家（1,261 个，覆盖 93,509 单）。

指标定义：

- 差评率 = `review_score <= 2` 的订单数 / **有评论**的订单数（分母不含有评论缺失的订单，避免被稀释）
- 分级：`NTILE(4) OVER (ORDER BY bad_review_rate)`，1 = 差评率最低，4 = 差评率最高
- 分组对比：订单量、平均发货时长、平均订单金额（`price + freight_value`，含运费）

预期结论：差评率最低的四分之一卖家平均 57.7 小时发货，最高的一组 109.1 小时（慢 89%），
差评率 5.03% vs 24.05% —— 履约时效与差评率强正相关。

In [13]:
from pathlib import Path

# 显式指定 UTF-8：Windows 默认 GBK，直接用 %sql --file 会因文件里的中文注释报 UnicodeDecodeError
seller_sql = Path("sql/02_seller_analysis.sql").read_text(encoding="utf-8")

# 文件内是「2 个视图的 DROP + CREATE」加末尾一条最终查询，交给 JupySQL 一次性执行
# 执行后 data/olist.db 会新增 v_seller_base（订单×卖家明细）和 v_seller_metrics（卖家指标 + 四分位）
%sql {{ seller_sql }}

# 后面两个单元格复用的分组对比查询（放在这里定义，避免重复贴 SQL）
tier_query = """
SELECT m.quality_tier                                   AS tier,
       COUNT(DISTINCT m.seller_id)                      AS sellers,
       COUNT(b.order_id)                                AS orders,
       ROUND(AVG(CASE WHEN b.ship_hours BETWEEN 0 AND 1440
                      THEN b.ship_hours END), 2)        AS avg_ship_hours,
       ROUND(AVG(b.pair_amount), 2)                     AS avg_order_value,
       ROUND(100.0 * SUM(CASE WHEN b.review_score <= 2 THEN 1 ELSE 0 END)
             / NULLIF(COUNT(b.review_score), 0), 2)     AS bad_rate_pct
FROM v_seller_metrics m
JOIN v_seller_base b ON b.seller_id = m.seller_id
GROUP BY m.quality_tier
ORDER BY m.quality_tier
"""

Running query in 'sqlite:///data/olist.db'

In [14]:
# JupySQL 默认只显示 10 行；这里只有 4 行，仍统一放宽，避免和其他查询表现不一致
%config SqlMagic.displaylimit = 100

# 最终对比：4 个差评率等级 ×（卖家数 / 订单量 / 平均发货时长 / 平均订单金额 / 差评率）
%sql {{ tier_query }}

Running query in 'sqlite:///data/olist.db'

tier,sellers,orders,avg_ship_hours,avg_order_value,bad_rate_pct
1,316,12449,57.72,160.26,5.03
2,315,29341,64.05,164.61,10.57
3,315,32260,76.83,133.71,15.27
4,315,19459,109.07,163.28,24.05


In [15]:
# 把查询结果转成 pandas DataFrame 打印（写报告、做图都用这一步）
result = %sql {{ tier_query }}     # %sql 返回的是 JupySQL 的 ResultSet，不能直接当 DataFrame 用
tier_df = result.DataFrame()      # 转成 pandas DataFrame

print("各差评率等级的履约与金额对比：")
print(tier_df.to_string(index=False))

# 组间差距：最快 vs 最慢，用于支撑"履约时效影响差评率"的结论
fast, slow = tier_df.iloc[0], tier_df.iloc[-1]
print(f"\n发货时长：tier1 {fast.avg_ship_hours} 小时 → tier4 {slow.avg_ship_hours} 小时"
      f"（慢了 {slow.avg_ship_hours / fast.avg_ship_hours - 1:.1%}）")
print(f"差评率：  tier1 {fast.bad_rate_pct}% → tier4 {slow.bad_rate_pct}%")

# 卖家级相关系数：验证"发货越慢、差评越多"在个体层面是否同样成立
seller_result = %sql SELECT avg_ship_hours, bad_review_rate, total_orders FROM v_seller_metrics
seller_df = seller_result.DataFrame()
print(f"\n卖家级相关系数（发货时长 vs 差评率）："
      f"{seller_df.avg_ship_hours.corr(seller_df.bad_review_rate):.3f}"
      f"（样本 {seller_df.avg_ship_hours.notna().sum()} 个卖家）")

Running query in 'sqlite:///data/olist.db'

各差评率等级的履约与金额对比：
 tier  sellers  orders  avg_ship_hours  avg_order_value  bad_rate_pct
    1      316   12449           57.72           160.26          5.03
    2      315   29341           64.05           164.61         10.57
    3      315   32260           76.83           133.71         15.27
    4      315   19459          109.07           163.28         24.05

发货时长：tier1 57.72 小时 → tier4 109.07 小时（慢了 89.0%）
差评率：  tier1 5.03% → tier4 24.05%


Running query in 'sqlite:///data/olist.db'


卖家级相关系数（发货时长 vs 差评率）：0.436（样本 1261 个卖家）
